In [ ]:
!pip install faker

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os
import json
from datetime import datetime, timedelta

In [ ]:
fake = Faker()
Faker.seed(42)
np.random.seed(42)

# Configuration
NUM_ADVISORS = 50
NUM_CLIENTS = 5000
NUM_FUNDS = 20
NUM_MONTHS = 24
OUTPUT_DIR = "mock_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Starting unified enterprise data generation...")

# ==========================================
# 1. GENERATE DIMENSIONS & SUB-DIMENSIONS
# ==========================================

# Dim_Region
regions = [
    {'Region_ID': 'REG_NA', 'Region_Name': 'North America', 'Regional_VP': fake.name()},
    {'Region_ID': 'REG_EMEA', 'Region_Name': 'Europe & Middle East', 'Regional_VP': fake.name()},
    {'Region_ID': 'REG_APAC', 'Region_Name': 'Asia Pacific', 'Regional_VP': fake.name()}
]
df_regions = pd.DataFrame(regions)
df_regions.to_csv(f"{OUTPUT_DIR}/Dim_Region.csv", index=False)

# Dim_Branch
branches = []
for i in range(1, 6):
    branches.append({
        'Branch_ID': f'BR_{i:02}',
        'Region_ID': random.choice(['REG_NA', 'REG_EMEA', 'REG_APAC']),
        'Branch_Name': fake.city() + ' Branch',
        'Branch_Manager': fake.name()
    })
df_branches = pd.DataFrame(branches)
df_branches.to_csv(f"{OUTPUT_DIR}/Dim_Branch.csv", index=False)

# Dim_Advisor
advisors = []
for i in range(NUM_ADVISORS):
    advisors.append({
        'Advisor_ID': f'ADV_{i+1:03}',
        'Advisor_Name': fake.name(),
        'Branch_ID': random.choice(df_branches['Branch_ID'].tolist()),
        'Hire_Date': fake.date_between(start_date='-10y', end_date='-2y').strftime('%Y-%m-%d'),
        'Capacity_Status': random.choices(['Low', 'Medium', 'High'], weights=[0.2, 0.5, 0.3])[0]
    })
df_advisors = pd.DataFrame(advisors)
df_advisors.to_csv(f"{OUTPUT_DIR}/Dim_Advisor.csv", index=False)

# Dim_Client
clients = []
for i in range(NUM_CLIENTS):
    adv = df_advisors.sample(1, weights=df_advisors['Capacity_Status'].map({'Low': 1, 'Medium': 3, 'High': 8})).iloc[0]
    target_retirement = fake.date_between(start_date='+5y', end_date='+30y').year
    if random.random() < 0.05:
        target_retirement = np.nan # 5% null rate for realism
        
    clients.append({
        'Client_ID': f'CLI_{i+1:05}',
        'Advisor_ID': adv['Advisor_ID'],
        'Client_Name': fake.name(),
        'Risk_Tolerance': random.choice(['Conservative', 'Moderate', 'Aggressive']),
        'Target_Retirement_Year': target_retirement,
        'Churn_Risk_Score': round(random.uniform(0.1, 0.9), 2) if adv['Capacity_Status'] == 'High' else round(random.uniform(0.01, 0.4), 2)
    })
df_clients = pd.DataFrame(clients)
df_clients.to_csv(f"{OUTPUT_DIR}/Dim_Client.csv", index=False)

# Dim_Account
accounts = []
for _, client in df_clients.iterrows():
    for _ in range(random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]):
        accounts.append({
            'Account_ID': fake.unique.bban(),
            'Client_ID': client['Client_ID'],
            'Account_Type': random.choice(['Individual Brokerage', 'Traditional IRA', 'Roth IRA', 'Joint Trust']),
            'Open_Date': fake.date_between(start_date='-5y', end_date='-2y').strftime('%Y-%m-%d')
        })
df_accounts = pd.DataFrame(accounts)
df_accounts.to_csv(f"{OUTPUT_DIR}/Dim_Account.csv", index=False)

# Dim_Asset_Class
asset_classes = [
    {'Asset_Class_ID': 'AC_01', 'Class_Name': 'Equity', 'Risk_Tier': 'High'},
    {'Asset_Class_ID': 'AC_02', 'Class_Name': 'Fixed Income', 'Risk_Tier': 'Low'},
    {'Asset_Class_ID': 'AC_03', 'Class_Name': 'Alternative', 'Risk_Tier': 'High'},
    {'Asset_Class_ID': 'AC_04', 'Class_Name': 'Cash', 'Risk_Tier': 'Low'}
]
df_asset_classes = pd.DataFrame(asset_classes)
df_asset_classes.to_csv(f"{OUTPUT_DIR}/Dim_Asset_Class.csv", index=False)

# Dim_Fund
funds = []
for i in range(NUM_FUNDS):
    ac = random.choice(asset_classes)
    funds.append({
        'Fund_ID': f'FND_{i+1:03}',
        'Fund_Name': fake.company() + ' Fund',
        'Ticker': fake.unique.lexify(text='????').upper(),
        'Asset_Class_ID': ac['Asset_Class_ID'],
        'Expense_Ratio': round(random.uniform(0.05, 1.50), 2)
    })
df_funds = pd.DataFrame(funds)
df_funds.to_csv(f"{OUTPUT_DIR}/Dim_Fund.csv", index=False)

# Dim_Date
start_date = datetime.now() - timedelta(days=NUM_MONTHS*30)
dates = [{
    'Date_ID': (start_date + timedelta(days=x)).strftime('%Y-%m-%d'),
    'Date': (start_date + timedelta(days=x)).strftime('%Y-%m-%d'),
    'Month': (start_date + timedelta(days=x)).month,
    'Quarter': ((start_date + timedelta(days=x)).month - 1) // 3 + 1,
    'Year': (start_date + timedelta(days=x)).year
} for x in range(NUM_MONTHS*30)]
df_dates = pd.DataFrame(dates)
df_dates.to_csv(f"{OUTPUT_DIR}/Dim_Date.csv", index=False)

# ==========================================
# 2. GENERATE FACT TABLES (With Skew)
# ==========================================
print("Generating Fact Tables...")
aum_records = []
trades = []

for month in range(NUM_MONTHS):
    current_date = start_date + timedelta(days=month*30)
    is_stagnation_period = month >= 12
    
    # Generate AUM (Monthly Snapshots)
    for _, client in df_clients.sample(frac=0.3).iterrows():
        adv_capacity = df_advisors[df_advisors['Advisor_ID'] == client['Advisor_ID']]['Capacity_Status'].values[0]
        fund = df_funds.sample(1).iloc[0]
        base_flow = random.uniform(-5000, 15000)
        
        if is_stagnation_period:
            base_flow -= random.uniform(5000, 25000) if adv_capacity == 'High' else random.uniform(0, 5000)
                
        aum_records.append({
            'AUM_ID': fake.uuid4(),
            'Client_ID': client['Client_ID'],
            'Fund_ID': fund['Fund_ID'],
            'Date_ID': current_date.strftime('%Y-%m-%d'),
            'EOM_Total_Value': round(max(0, random.uniform(10000, 500000) + base_flow), 2),
            'Net_Flow': round(base_flow, 2)
        })
        
    # Generate Trades (Daily)
    for _, account in df_accounts.sample(frac=0.1).iterrows():
        client = df_clients[df_clients['Client_ID'] == account['Client_ID']].iloc[0]
        adv_capacity = df_advisors[df_advisors['Advisor_ID'] == client['Advisor_ID']]['Capacity_Status'].values[0]
        fund = df_funds.sample(1).iloc[0]
        
        trade_type = random.choices(['Buy', 'Sell'], weights=[0.3, 0.7])[0] if (is_stagnation_period and adv_capacity == 'High') else random.choices(['Buy', 'Sell'], weights=[0.7, 0.3])[0]
        qty = round(random.uniform(10, 500), 4)
        price = round(random.uniform(20, 250), 2)
        
        trades.append({
            'Trade_ID': fake.uuid4(),
            'Account_ID': account['Account_ID'],
            'Fund_ID': fund['Fund_ID'],
            'Date_ID': (current_date + timedelta(days=random.randint(0, 27))).strftime('%Y-%m-%d'),
            'Trade_Type': trade_type,
            'Quantity': qty,
            'Price': price,
            'Total_Value': round(qty * price, 2),
            'Status': random.choices(['Settled', 'Pending', 'Failed'], weights=[0.95, 0.04, 0.01])[0]
        })

pd.DataFrame(aum_records).to_csv(f"{OUTPUT_DIR}/Fact_AUM_Monthly.csv", index=False)
pd.DataFrame(trades).to_csv(f"{OUTPUT_DIR}/Fact_Trades.csv", index=False)

# ==========================================
# 3. GENERATE UNSTRUCTURED DOCS (JSON)
# ==========================================
print("Generating Unstructured Documents...")

# CRM Notes
crm_notes = []
for _, client in df_clients.sample(100).iterrows(): 
    crm_notes.append({
        'Note_ID': fake.uuid4(),
        'Client_ID': client['Client_ID'],
        'Advisor_ID': client['Advisor_ID'],
        'Interaction_Date': fake.date_between(start_date='-1y', end_date='today').strftime('%Y-%m-%d'),
        'Note_Text': f"Met with {client['Client_Name']}. Discussed their target retirement in {client['Target_Retirement_Year']}. Concerned about market volatility."
    })
with open(f"{OUTPUT_DIR}/Doc_CRM_Notes.json", 'w') as f: json.dump(crm_notes, f, indent=4)

# Prospectuses
prospectuses = []
for _, fund in df_funds.iterrows():
    prospectuses.append({
        'Chunk_ID': fake.uuid4(),
        'Fund_ID': fund['Fund_ID'],
        'Doc_Type': 'Prospectus',
        'Text_Chunk': f"The {fund['Fund_Name']} ({fund['Ticker']}) seeks long-term capital appreciation amidst macroeconomic headwinds."
    })
with open(f"{OUTPUT_DIR}/Doc_Fund_Prospectus.json", 'w') as f: json.dump(prospectuses, f, indent=4)

# Transcripts
transcripts = []
for _, client in df_clients.sample(50).iterrows(): 
    adv_capacity = df_advisors[df_advisors['Advisor_ID'] == client['Advisor_ID']]['Capacity_Status'].values[0]
    sentiment = "Client expressed frustration regarding lack of proactive communication." if adv_capacity == 'High' else f"Client is satisfied with current allocation regarding {client['Risk_Tolerance']} risk tolerance."
    transcripts.append({
        'Transcript_ID': fake.uuid4(),
        'Client_ID': client['Client_ID'],
        'Advisor_ID': client['Advisor_ID'],
        'Call_Date': fake.date_between(start_date='-6m', end_date='today').strftime('%Y-%m-%d'),
        'Transcript_Text': f"[Automated Transcription]: {sentiment}"
    })
with open(f"{OUTPUT_DIR}/Doc_Call_Transcripts.json", 'w') as f: json.dump(transcripts, f, indent=4)

print(f"Success! All 13 files generated in ./{OUTPUT_DIR}/")